In [1]:
'''
PROBLEM STATEMENT:
Write code to simulate requests coming from clients and
distribute them among the servers using load balancing algorithms.
'''

import time
import random


# =====================================================
# ROUND ROBIN
# =====================================================
class RoundRobin:

    def __init__(self, servers):

        self.servers = servers
        self.current_index = -1

    def get_next_server(self):

        self.current_index = (
            self.current_index + 1
        ) % len(self.servers)

        return self.servers[self.current_index]


# =====================================================
# WEIGHTED ROUND ROBIN
# =====================================================
class WeightedRoundRobin:

    def __init__(self, servers, weights):

        self.servers = servers
        self.weights = weights

        self.current_index = -1
        self.current_weight = 0

    def get_next_server(self):

        while True:

            self.current_index = (
                self.current_index + 1
            ) % len(self.servers)

            if self.current_index == 0:

                self.current_weight -= 1

                if self.current_weight <= 0:
                    self.current_weight = max(self.weights)

            if self.weights[self.current_index] >= self.current_weight:

                return self.servers[self.current_index]


# =====================================================
# LEAST CONNECTIONS
# =====================================================
class LeastConnections:

    def __init__(self, servers):

        self.servers = {
            server: 0 for server in servers
        }

    def get_next_server(self):

        min_connections = min(
            self.servers.values()
        )

        least_loaded = [

            server

            for server, count
            in self.servers.items()

            if count == min_connections
        ]

        selected = random.choice(least_loaded)

        self.servers[selected] += 1

        return selected

    def release_connection(self, server):

        if self.servers[server] > 0:

            self.servers[server] -= 1


# =====================================================
# LEAST RESPONSE TIME
# =====================================================
class LeastResponseTime:

    def __init__(self, servers):

        self.servers = servers

        self.response_times = [
            0
        ] * len(servers)

    def get_next_server(self):

        min_time = min(self.response_times)

        index = self.response_times.index(min_time)

        return self.servers[index]

    def update_response_time(
        self,
        server,
        response_time
    ):

        index = self.servers.index(server)

        self.response_times[index] = response_time


# =====================================================
# HELPER FUNCTION
# =====================================================
def simulate_response_time():

    delay = random.uniform(0.1, 1.0)

    time.sleep(delay)

    return delay


# =====================================================
# DEMO FUNCTION
# =====================================================
def demonstrate_algorithm(
    name,
    lb,
    iterations=6,
    use_response_time=False,
    use_connections=False
):

    print("\n===================================")
    print(name)
    print("===================================")

    for i in range(iterations):

        server = lb.get_next_server()

        print(f"Request {i+1} --> {server}")

        # Response Time Handling
        if use_response_time:

            rt = simulate_response_time()

            lb.update_response_time(server, rt)

            print(f"Response Time : {rt:.2f} sec")

        # Release Connections
        if use_connections:

            lb.release_connection(server)


# =====================================================
# MAIN PROGRAM
# =====================================================

servers = [
    "Server1",
    "Server2",
    "Server3"
]


# ---------------- ROUND ROBIN ----------------
rr = RoundRobin(servers)

demonstrate_algorithm(
    "ROUND ROBIN",
    rr
)


# ---------------- WEIGHTED ROUND ROBIN ----------------
weights = [5, 1, 1]

wrr = WeightedRoundRobin(
    servers,
    weights
)

demonstrate_algorithm(
    "WEIGHTED ROUND ROBIN",
    wrr,
    iterations=7
)


# ---------------- LEAST CONNECTIONS ----------------
lc = LeastConnections(servers)

demonstrate_algorithm(
    "LEAST CONNECTIONS",
    lc,
    use_connections=True
)


# ---------------- LEAST RESPONSE TIME ----------------
lrt = LeastResponseTime(servers)

demonstrate_algorithm(
    "LEAST RESPONSE TIME",
    lrt,
    use_response_time=True
)


ROUND ROBIN
Request 1 --> Server1
Request 2 --> Server2
Request 3 --> Server3
Request 4 --> Server1
Request 5 --> Server2
Request 6 --> Server3

WEIGHTED ROUND ROBIN
Request 1 --> Server1
Request 2 --> Server1
Request 3 --> Server1
Request 4 --> Server1
Request 5 --> Server1
Request 6 --> Server2
Request 7 --> Server3

LEAST CONNECTIONS
Request 1 --> Server2
Request 2 --> Server1
Request 3 --> Server2
Request 4 --> Server3
Request 5 --> Server3
Request 6 --> Server2

LEAST RESPONSE TIME
Request 1 --> Server1
Response Time : 0.70 sec
Request 2 --> Server2
Response Time : 0.53 sec
Request 3 --> Server3
Response Time : 0.99 sec
Request 4 --> Server2
Response Time : 0.58 sec
Request 5 --> Server2
Response Time : 1.00 sec
Request 6 --> Server1
Response Time : 0.87 sec
